# Pharmacogenomic lookups with DeepDNA: REST and MCP

A short, runnable tour of the DeepDNA knowledge endpoints, for teaching and for people building agents.

DeepDNA serves curated, cited records on genes, variants and drug–gene pairs. A drug–gene record says what its source (a CPIC or DPWG guideline, a drug label or, where neither applies, a study) states for a phenotype group, and links to that source. The same records are available three ways:

- REST: `https://deepdna.ai/api/v1` ([docs](https://deepdna.ai/docs/), [OpenAPI](https://deepdna.ai/openapi.json))
- A remote MCP server: `https://deepdna.ai/mcp`
- The local package in this repository, `deepdna-mcp`

No key and no account are needed for the REST API or the remote MCP server.

**Educational information only, not medical advice.** The endpoints take a gene symbol, an rsID or a drug name. They have no field for a genotype, so this notebook never sends one: it asks what a source says about a phenotype group such as "poor metabolizer", not about any person. Records change as guidelines are updated; every response carries a `data_as_of` timestamp, and the outputs saved below are what the API returned on the day the notebook was last run.

Requirements: `pip install requests pandas`

In [1]:
import json
import os

import pandas as pd
import requests

API = "https://deepdna.ai/api/v1"
MCP = "https://deepdna.ai/mcp"

http = requests.Session()
# Say who you are: a descriptive User-Agent helps the operator tell clients apart.
http.headers["User-Agent"] = os.environ.get("DEEPDNA_UA", "deepdna-examples-notebook/1.0")

pd.set_option("display.max_colwidth", 80)


def get(path):
    """GET a REST path; return (status code, parsed JSON). Never raises on 404."""
    r = http.get(API + path, timeout=30)
    return r.status_code, r.json()

## 1. List the genes

`GET /genes` returns every gene in the knowledge base with a category and a one-line summary.

In [2]:
status, body = get("/genes")
genes = pd.DataFrame(body["genes"])
print(status, "-", body["count"], "genes, data as of", body["meta"]["data_as_of"])
genes[["symbol", "name", "category"]]

200 - 17 genes, data as of 2026-11-08T12:09:46.225Z


,symbol,name,category
0,MTHFR,"5,10-Methylenetetrahydrofolate reductase",nutrigenomics
1,COMT,Catechol-O-methyltransferase,nutrigenomics
2,CYP2D6,Cytochrome P450 Family 2 Subfamily D Member 6,pharmacogene
3,CYP2C19,Cytochrome P450 Family 2 Subfamily C Member 19,pharmacogene
4,CYP2C9,Cytochrome P450 Family 2 Subfamily C Member 9,pharmacogene
5,SLCO1B1,Solute Carrier Organic Anion Transporter Family Member 1B1,pharmacogene
6,G6PD,Glucose-6-Phosphate Dehydrogenase,pharmacogene
7,UGT1A1,UDP Glucuronosyltransferase Family 1 Member A1,pharmacogene
8,HFE,Homeostatic Iron Regulator,disease_risk
9,APOE,Apolipoprotein E,disease_risk


## 2. Look up one gene

`GET /genes/{symbol}` returns the gene's function, its phenotype groups, key variants, the drugs curated for it and the sources.

In [3]:
status, body = get("/genes/CYP2C19")
gene = body["gene"]

print(gene["symbol"], "-", gene["name"])
print()
print(gene["summary"])
print()
print("Phenotype groups:")
for p in gene["phenotypes"]:
    print("  -", p)

CYP2C19 - Cytochrome P450 Family 2 Subfamily C Member 19

CYP2C19 is a hepatic cytochrome P450 enzyme that activates the antiplatelet prodrug clopidogrel and clears proton-pump inhibitors, several antidepressants, diazepam and the antifungal voriconazole. Poor metabolizers convert little clopidogrel to its active form, a difference linked to higher rates of stent thrombosis and recurrent cardiovascular events and the basis of the drug's FDA boxed warning. Star-allele haplotypes map to poor, intermediate, normal, rapid and ultra-rapid metabolizer phenotypes.

Phenotype groups:
  - Poor metabolizer (PM): two no-function alleles (e.g. *2/*2, *2/*3)
  - Intermediate metabolizer (IM): one no-function allele with one normal or increased-function allele (e.g. *1/*2, *2/*17)
  - Normal metabolizer (NM): two normal-function alleles (*1/*1)
  - Rapid metabolizer (RM): one normal and one increased-function allele (*1/*17)
  - Ultra-rapid metabolizer (UM): two increased-function alleles (*17/*17)


In [4]:
pd.DataFrame(gene["drugs"])[["drug", "guideline", "level"]]

,drug,guideline,level
0,clopidogrel,CPIC,CPIC A
1,omeprazole,CPIC,CPIC A
2,lansoprazole,CPIC,CPIC A
3,pantoprazole,CPIC,CPIC A
4,citalopram,CPIC,CPIC A
5,escitalopram,CPIC,CPIC A
6,sertraline,CPIC,CPIC A
7,voriconazole,CPIC,CPIC A
8,amitriptyline,CPIC,CPIC A
9,dexlansoprazole,CPIC,CPIC B


## 3. Look up one drug–gene pair

`GET /pgx/{gene}/{drug}` returns one record. The fields to show a reader are:

- `guideline`: what the record rests on (`CPIC`, `DPWG`, `drug label`, a guideline plus a label such as `DPWG; FDA label`, or `null` when it rests on the primary literature).
- `level`: CPIC's level for the pair, copied from the CPIC database, or `null` when CPIC has not rated the pair. It rates the pair, not the strength of a single recommendation; the [docs](https://deepdna.ai/docs/#pgx) explain the scale.
- `summary`: what the source recommends, by phenotype group.
- `sources`: where it comes from, each with a link. Quote them.

In [5]:
status, body = get("/pgx/CYP2C19/clopidogrel")
pair = body["pair"]

print("Pair:     ", pair["gene"], "/", pair["drug"])
print("Guideline:", pair["guideline"])
print("Level:    ", pair["level"])
print()
print(pair["summary"])
print()
print("Phenotype groups:")
for p in pair["phenotypes"]:
    print("  -", p)
print()
print("Sources:")
for s in pair["sources"]:
    print("  -", s["label"])
    print("   ", s["url"])
print()
print(body["meta"]["disclaimer"])
print("Data as of", body["meta"]["data_as_of"])

Pair:      CYP2C19 / clopidogrel
Guideline: CPIC
Level:     CPIC A

Prodrug activated by CYP2C19. CPIC 2022 recommends an alternative antiplatelet (prasugrel or ticagrelor) for poor and intermediate metabolizers undergoing PCI for acute coronary syndromes. FDA boxed warning about reduced effectiveness in poor metabolizers.

Phenotype groups:
  - Poor metabolizer (PM): two no-function alleles (e.g. *2/*2, *2/*3)
  - Intermediate metabolizer (IM): one no-function allele with one normal or increased-function allele (e.g. *1/*2, *2/*17)
  - Normal metabolizer (NM): two normal-function alleles (*1/*1)
  - Rapid metabolizer (RM): one normal and one increased-function allele (*1/*17)
  - Ultra-rapid metabolizer (UM): two increased-function alleles (*17/*17)

Sources:
  - CPIC — CYP2C19 gene page
    https://cpicpgx.org/gene/cyp2c19/
  - PharmGKB CYP2C19 (PA124)
    https://www.pharmgkb.org/gene/PA124
  - PharmVar CYP2C19
    https://www.pharmvar.org/gene/CYP2C19
  - Lee et al., Clin Pharmacol

## 4. Check a medication list

A common question for an agent is "which of these drugs have a curated drug–gene record at all?". The MCP tool `check_medications` answers it in one call. It takes generic drug names and nothing else: no genotype and no patient details. The list below is an arbitrary example, not anyone's prescription.

The REST API has no equivalent endpoint, so this cell calls the MCP server with a plain HTTP POST. Section 6 walks through what that request is.

In [6]:
MCP_HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def mcp_call(tool, arguments, request_id=1):
    """Call one MCP tool with a single JSON-RPC request; return the `result` object."""
    payload = {"jsonrpc": "2.0", "id": request_id, "method": "tools/call",
               "params": {"name": tool, "arguments": arguments}}
    r = http.post(MCP, headers=MCP_HEADERS, json=payload, timeout=30)
    return r.json()["result"]


medications = ["clopidogrel", "omeprazole", "simvastatin", "codeine", "metformin"]
checked = mcp_call("check_medications", {"medications": medications})["structuredContent"]

rows = [
    {"drug": hit["drug"], "gene": g["gene"], "basis": g["basis"]}
    for hit in checked["with_curated_records"]
    for g in hit["genes"]
]
print("Not covered:", checked["not_covered"])
print()
print(checked["coverage_note"])
pd.DataFrame(rows)

Not covered: ['metformin']

Covers 84 curated drug–gene pairs across 16 genes, by generic name. basis says what each record rests on: a CPIC or DPWG guideline, a drug label, or the primary literature when no guideline recommendation or drug label applies. A drug listed under not_covered is not in this knowledge base; that does not mean it has no pharmacogenomic interaction. This lists what each record's source says; it does not say what this person should take.


,drug,gene,basis
0,clopidogrel,CYP2C19,CPIC guideline
1,omeprazole,CYP2C19,CPIC guideline
2,simvastatin,SLCO1B1,CPIC guideline
3,codeine,CYP2D6,CPIC guideline


"Not covered" means the drug is not in this knowledge base. As the coverage note says, it does not mean the drug has no pharmacogenomic interaction. The note counts 16 genes because one of the 17 genes in section 1 (HFE) has no drug–gene pairs.

## 5. A pair that does not exist

Asking for a pair with no record returns HTTP 404. The body carries an `error` object, with a machine-readable `code` and a `hint` that lists what *is* curated for that gene, next to the usual `meta` block.

In [7]:
status, body = get("/pgx/VKORC1/acenocoumarol")
print(status)
print(json.dumps(body["error"], indent=2, ensure_ascii=False))

404
{
  "code": "not_found",
  "resource": "pgx",
  "id": "VKORC1/acenocoumarol",
  "hint": "Drugs curated for VKORC1: warfarin"
}


How to handle it: treat a 404 as "there is no record", say so, and stop. Do not let your code, or the language model behind your agent, fill the gap from memory, and do not reuse the record of a related drug as if it applied. The `hint` is there so that an agent can recover with one more call (for example, to correct a misspelt drug name) instead of guessing.

In [8]:
def lookup_pair(gene, drug):
    """Return the record for a drug-gene pair, or an explicit 'no record' answer."""
    status, body = get(f"/pgx/{gene}/{drug}")
    if status == 404:
        return {"found": False,
                "message": f"DeepDNA has no curated record for {gene} and {drug}.",
                "hint": body["error"]["hint"]}
    pair = body["pair"]
    return {"found": True, "guideline": pair["guideline"], "level": pair["level"],
            "summary": pair["summary"], "sources": pair["sources"]}


for gene_symbol, drug in [("VKORC1", "warfarin"), ("VKORC1", "acenocoumarol")]:
    answer = lookup_pair(gene_symbol, drug)
    if answer["found"]:
        print(f"{gene_symbol} / {drug}: record found ({answer['guideline']}, {answer['level']}),",
              len(answer["sources"]), "sources")
    else:
        print(f"{gene_symbol} / {drug}: {answer['message']} Hint: {answer['hint']}")

VKORC1 / warfarin: record found (CPIC, CPIC A), 2 sources


VKORC1 / acenocoumarol: DeepDNA has no curated record for VKORC1 and acenocoumarol. Hint: Drugs curated for VKORC1: warfarin


## 6. The same lookup over MCP, with raw JSON-RPC

MCP over streamable HTTP is JSON-RPC 2.0 sent with POST, so no SDK is needed to see what an assistant does. The exchange is `initialize`, then `tools/list` to discover the tools, then `tools/call`.

In [9]:
init = http.post(MCP, headers=MCP_HEADERS, timeout=30, json={
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-06-18", "capabilities": {},
               "clientInfo": {"name": "deepdna-examples-notebook", "version": "1.0"}},
})
server = init.json()["result"]
print("Server:  ", server["serverInfo"]["name"], server["serverInfo"]["version"])
print("Protocol:", server["protocolVersion"])
print()
print(server["instructions"])

Server:   deepdna 1.3.0
Protocol: 2025-06-18

DeepDNA (https://deepdna.ai): cited, curated knowledge about genes, variants (by rsID) and drug–gene pairs (CPIC or DPWG guidance where a guideline exists) for the genes listed by list_genes. Quote the sources it returns. check_medications screens a medication list for drugs with curated drug–gene records. Never send a genotype or a DNA file: the tools take gene symbols, rsIDs and drug names only, any other argument is refused and not used. Never type a personal genetic result into an argument. DeepDNA never assigns a person a phenotype, metabolizer status, risk or dose. A drug–gene record states what its source (a guideline, a drug label or, where no guideline makes a genotype-based recommendation and no label applies, a study) says for the phenotype groups or genotypes it covers, dose adjustments included; a study-based record reports a research finding, not a recommendation. Present it as that source's statement and do not suggest starti

The `instructions` string is what the server tells a model about how to use it. The server is stateless: `initialize` may return an `Mcp-Session-Id` header, and sending it back on later requests is optional (the cell below does).

In [10]:
session_id = init.headers.get("Mcp-Session-Id")
if session_id:
    MCP_HEADERS["Mcp-Session-Id"] = session_id

listed = http.post(MCP, headers=MCP_HEADERS, timeout=30,
                   json={"jsonrpc": "2.0", "id": 2, "method": "tools/list"}).json()["result"]
pd.DataFrame([{"tool": t["name"], "arguments": ", ".join(t["inputSchema"]["properties"]) or "(none)"}
              for t in listed["tools"]])

,tool,arguments
0,list_genes,(none)
1,lookup_gene,symbol
2,lookup_variant,rsid
3,lookup_drug_gene,"gene, drug"
4,check_medications,medications


In [11]:
result = mcp_call("lookup_drug_gene", {"gene": "CYP2C19", "drug": "clopidogrel"}, request_id=3)
mcp_pair = result["structuredContent"]["pair"]

print("Guideline:", mcp_pair["guideline"], "| Level:", mcp_pair["level"])
print(mcp_pair["summary"])
print()
print("Same record as the REST answer in section 3:", mcp_pair == pair)

Guideline:

 CPIC | Level: CPIC A
Prodrug activated by CYP2C19. CPIC 2022 recommends an alternative antiplatelet (prasugrel or ticagrelor) for poor and intermediate metabolizers undergoing PCI for acute coronary syndromes. FDA boxed warning about reduced effectiveness in poor metabolizers.

Same record as the REST answer in section 3: True


A pair with no record comes back as a tool result with `isError: true` and a text the model can read, the MCP counterpart of the 404 in section 5.

In [12]:
missing = mcp_call("lookup_drug_gene", {"gene": "VKORC1", "drug": "acenocoumarol"}, request_id=4)
print("isError:", missing.get("isError"))
print(missing["content"][0]["text"])

isError: True
No guidance curated for VKORC1 and "acenocoumarol". Curated drugs: warfarin.


## 7. Connect it to an assistant

An MCP client does the exchange above for you. Per-client instructions, with one-click links for some of them, are at <https://deepdna.ai/connect/>. For example, in Claude Code:

```bash
claude mcp add --transport http deepdna https://deepdna.ai/mcp
```

and in clients that read an `mcpServers` file:

```json
{
  "mcpServers": {
    "deepdna": { "type": "http", "url": "https://deepdna.ai/mcp" }
  }
}
```

Then ask, for example: *"What does CPIC say about clopidogrel for CYP2C19 poor metabolizers? Quote the sources."*

Whichever route you use, the same rules apply: present a record as its source's statement, keep the links, say "there is no record" when there is none, and leave any decision about a person's medication to the prescriber.

- API docs: <https://deepdna.ai/docs/>
- Teaching cases in Spanish built on the same calls: [casos_es.md](casos_es.md)